# SDLC BVE Dashboards for X — Copilot Command Workbook

This notebook is a **Copilot-led onboarding workbook** for new users.

Each step below runs GitHub Copilot CLI from Python instead of raw `%%bash`, so notebook cells show the **real Copilot stdout/stderr** instead of an opaque Jupyter `CalledProcessError`.

## Current paths to prefer

- Use `.github/workflows/pipeline-deploy.yml` as the current pipeline workflow.
- Treat `.github/workflows/deploy-dashboards.yml` as legacy.
- Prefer current V4 dashboards when they exist, especially:
  - `dashboard/v4/ai-assisted-efficiency/`
  - `dashboard/v4/agentic-efficiency/`
- Treat older manual-upload and older-version dashboards as legacy unless you are intentionally working there.

## Source-of-truth docs

- `README.md`
- `docs/getting-started.md`
- `docs/pat-setup.md`
- `docs/data-collection.md`
- `docs/dashboard-status.md`
- `dependencies/README.md`

## Security note

Do **not** put secret PAT values into the notebook or tracked files. Let Copilot explain the setup, but create and store secrets outside the notebook.

## Step 0 — Load helpers and verify the environment

Run the next two cells first.

In [2]:
import os
from pathlib import Path
import shlex
import shutil
import subprocess


def _ensure_token_env():
    """Inherit GITHUB_TOKEN from gh CLI if the kernel doesn't have it."""
    if os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN'):
        return
    # Try to pull from gh auth
    r = subprocess.run(
        ['gh', 'auth', 'token'],
        capture_output=True, text=True, check=False,
    )
    if r.returncode == 0 and r.stdout.strip():
        os.environ['GITHUB_TOKEN'] = r.stdout.strip()
        print('ℹ️  Inherited GITHUB_TOKEN from `gh auth token`')
    else:
        print('⚠️  No GITHUB_TOKEN found. Run `gh auth login` in a terminal or set GITHUB_TOKEN.')


_ensure_token_env()


def repo_root():
    probe = subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True,
        text=True,
        check=False,
    )
    if probe.returncode == 0:
        return Path(probe.stdout.strip())
    return Path.cwd()


REPO_ROOT = repo_root()


def _print_section(title, body):
    if body:
        print(f"\n[{title}]")
        print(body.rstrip())


def run_cmd(command, *, cwd=REPO_ROOT, shell=False):
    printable = command if isinstance(command, str) else " ".join(shlex.quote(part) for part in command)
    print(f"$ {printable}")
    completed = subprocess.run(
        command,
        cwd=str(cwd),
        shell=shell,
        text=True,
        capture_output=True,
        check=False,
    )
    _print_section("stdout", completed.stdout)
    _print_section("stderr", completed.stderr)
    if completed.returncode != 0:
        print(f"\nExit code: {completed.returncode}")
    return completed


def explain_copilot_failure(stderr):
    message = (stderr or "").lower()
    if "not logged in" in message or "/login" in message or "authentication" in message:
        return "Copilot CLI needs authentication. Open a terminal in the repo, run `copilot`, then use `/login`. If your org requires token auth instead, set `GH_TOKEN` or `GITHUB_TOKEN` with a token that has Copilot Requests permission."
    if "not found" in message:
        return "Copilot CLI is not installed in this environment. Rebuild the devcontainer or run the optional install cell."
    if "rate limit" in message:
        return "The request was rate-limited. Wait a moment, then rerun the cell."
    return "Read stderr above for the exact Copilot CLI error, fix that issue, then rerun the cell."


def run_copilot(prompt, *, cwd=REPO_ROOT):
    if shutil.which("copilot") is None:
        print("Copilot CLI is not installed. Run the optional install cell or rebuild the devcontainer.")
        return 127

    command = [
        "copilot",
        "--prompt",
        prompt,
        "--silent",
        "--no-color",
        "--allow-all",
    ]
    result = run_cmd(command, cwd=cwd)
    if result.returncode != 0:
        print(f"\nCopilot exited with status {result.returncode}.")
        print(explain_copilot_failure(result.stderr))
    return result.returncode


print(f"Notebook repo root: {REPO_ROOT}")

Notebook repo root: /workspaces/SDLC-BVE-dashboards-for-X


In [7]:
run_cmd(
    "pwd && node --version && npm --version && python3 --version && gh --version | head -n 1 && (copilot --version || true) && (jupyter lab --version || true) && (gh auth status || true)",
    shell=True,
)


$ pwd && node --version && npm --version && python3 --version && gh --version | head -n 1 && (copilot --version || true) && (jupyter lab --version || true) && (gh auth status || true)

[stdout]
/workspaces/SDLC-BVE-dashboards-for-X
v22.22.2
10.9.7
Python 3.13.5
gh version 2.92.0 (2026-04-28)
GitHub Copilot CLI 1.0.44-2.
Run 'copilot update' to check for updates.
4.5.7
github.com
  ✓ Logged in to github.com account MattG57 (GITHUB_TOKEN)
  - Active account: true
  - Git operations protocol: https
  - Token: ghu_************************************


CompletedProcess(args='pwd && node --version && npm --version && python3 --version && gh --version | head -n 1 && (copilot --version || true) && (jupyter lab --version || true) && (gh auth status || true)', returncode=0, stdout="/workspaces/SDLC-BVE-dashboards-for-X\nv22.22.2\n10.9.7\nPython 3.13.5\ngh version 2.92.0 (2026-04-28)\nGitHub Copilot CLI 1.0.44-2.\nRun 'copilot update' to check for updates.\n4.5.7\n\x1bgithub.com\x1b\n  \x1b✓\x1b Logged in to github.com account \x1bMattG57\x1b (GITHUB_TOKEN)\n  - Active account: \x1btrue\x1b\n  - Git operations protocol: \x1bhttps\x1b\n  - Token: \x1bghu_************************************\x1b\n", stderr='')

If `copilot` is missing, uncomment and run the next cell, or rebuild the devcontainer first.

In [4]:
# Uncomment if Copilot CLI is missing:
# run_cmd("curl -fsSL https://gh.io/copilot-install | bash", shell=True)


## Step 1 — Ask Copilot to orient itself to the repo

In [8]:
run_copilot(
    """Read README.md, docs/getting-started.md, docs/pat-setup.md, docs/data-collection.md, docs/dashboard-status.md, and dependencies/README.md. Summarize the current setup path for a brand-new user of this repo, including the current workflow, the dashboards to prefer first, and what paths should be treated as legacy."""
)


$ copilot --prompt 'Read README.md, docs/getting-started.md, docs/pat-setup.md, docs/data-collection.md, docs/dashboard-status.md, and dependencies/README.md. Summarize the current setup path for a brand-new user of this repo, including the current workflow, the dashboards to prefer first, and what paths should be treated as legacy.' --silent --no-color --allow-all

Exit code: 1

Copilot exited with status 1.
Read stderr above for the exact Copilot CLI error, fix that issue, then rerun the cell.


1

## Step 2 — Ask Copilot to guide PAT setup

In [6]:
run_copilot(
    """Use docs/pat-setup.md and docs/getting-started.md to guide a new user through creating the correct GitHub Personal Access Token for this repo. Explain the required scopes, explain SSO authorization, and explain the safest next step for using the token locally or in GitHub Actions without asking the user to paste the secret into chat or into files."""
)


$ copilot --prompt 'Use docs/pat-setup.md and docs/getting-started.md to guide a new user through creating the correct GitHub Personal Access Token for this repo. Explain the required scopes, explain SSO authorization, and explain the safest next step for using the token locally or in GitHub Actions without asking the user to paste the secret into chat or into files.' --silent --no-color --allow-all

Exit code: 1

Copilot exited with status 1.
Read stderr above for the exact Copilot CLI error, fix that issue, then rerun the cell.


1

## Step 3 — Configure `query-settings.json`

Edit the placeholder values in the next code cell before running it.

In [ ]:
run_copilot(
    """Update query-settings.json for this repository using ORG='your-org', ENTERPRISE='your-enterprise-slug-or-empty', and DAYS='28'. Keep the change minimal, explain the final effective configuration, and do not modify unrelated files."""
)


## Step 4 — Configure `dashboard-config.json`

Edit the placeholder values in the next code cell before running it.

In [ ]:
run_copilot(
    """Update dashboard-config.json for this repository using cfg_total_developers=500, cfg_pct_time_coding=0.25, cfg_labor_cost_per_hour=100, est_hrs_per_kloc=1, and est_duration_factor=10. Explain what each of these values affects in the dashboards, keep the change minimal, and do not modify unrelated files."""
)


## Step 5 — Validate with a dry run

In [ ]:
run_copilot(
    """Run ./run-query.sh --dry-run, interpret the output, and fix any safe configuration problems you find in this repository. Do not ask for secrets in chat. If something is blocked by missing external setup, explain the exact next manual step."""
)


## Step 6 — Run local collection or materialization

In [ ]:
run_copilot(
    """Check whether local authentication is ready for this repo, then run the safest next command to execute the local pipeline. If authentication or PAT setup is incomplete, explain the exact external step that is still required before retrying."""
)


In [ ]:
run_copilot(
    """Run ./run-query.sh --materialize-only for this repository, explain what artifacts were refreshed, and report any issues clearly."""
)


## Step 7 — Serve dashboards locally

In [ ]:
run_copilot(
    """Serve the dashboards locally for this repository, tell me which local URL to open, and point me to the best current dashboards to inspect first. Prefer V4 dashboards where available and explain which legacy paths I can ignore for now."""
)


## Step 8 — Prepare GitHub Pages deployment

In [ ]:
run_copilot(
    """Help me prepare GitHub Pages deployment for this repository using .github/workflows/pipeline-deploy.yml. Check what still needs to be configured for DASHBOARD_GH_TOKEN, ORG or ENTERPRISE variables, DAYS, and Pages settings. Then give me the smallest safe next steps and the command to trigger the workflow."""
)


In [ ]:
run_copilot(
    """Trigger the current pipeline deploy workflow for this repository with data collection skipped, then report the workflow run link and current status."""
)


## Step 9 — Troubleshoot setup problems

In [ ]:
run_copilot(
    """Diagnose my setup problem for this repository. Check the environment, authentication state, config files, dry-run output, current workflow path, and devcontainer or Jupyter setup. Fix what you can directly, explain what is blocked, and tell me the exact next action if manual intervention is required."""
)


## Optional note on approvals

- This notebook uses `--allow-all` so Copilot can work cleanly inside notebook cells without separate tool, path, or URL approval prompts.
- If you prefer a narrower approval scope in a terminal, you can switch to `--allow-all-tools` or one or more `--allow-tool` flags.
- Interactive slash commands such as `/allow-all` and `/yolo` apply to interactive sessions, not this workbook format.

In [ ]:
run_cmd(
    "printf '%s\\n' README.md docs/getting-started.md docs/pat-setup.md docs/data-collection.md docs/dashboard-status.md dependencies/README.md",
    shell=True,
)
